## Matchup the chl dataset to OC-CCI chl 

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime,date, timedelta
import time
import warnings
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
import os

In [2]:
def haversine_distance_vectorized(lat1, lon1, lat2, lon2):
    """
    Calculate the haversine distance between two sets of points in a vectorized way.
    Parameters:
        lat1, lon1: Latitude and longitude of the first set of points
        lat2, lon2: Latitude and longitude of the second set of points 
    Returns:
        Array of distances in kilometers.
    """
    R = 6371  #radius of earth
    #convert latitude and longitude from degrees to radians
    lat1_rad, lon1_rad = np.radians(lat1), np.radians(lon1)
    lat2_rad, lon2_rad = np.radians(lat2), np.radians(lon2)

    #calculate differences in coordinates
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad

    #haversine formula
    a = np.sin(dlat / 2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = R * c
    return distance

In [3]:
def slice_before_concat(ds):
        """
        Slices the xarray dataset to keep only a specific lat/lon range instead of loading in global data
        NOTE: 'lat' and 'lon' here need to have same naming convention as the satellite nc file
        """
        ds = ds.sel(lat=slice(80,10),lon=slice(-179,-40)) #slice xarray within these lat and lon bounds 
        return ds

In [4]:
def insitu_year(df,year):
    ''' 
    Takes a dataframe and given year, returns dataframe with only data from that year and list of YYYmmdd times to pull xarray file names 
    df = in situ dataframe, with datetime column
    year = desired year
    returns:
    filtered_df= in situ dataframe subset to the given year
    dates = list of unique dates in format YYYYmmdd
    '''
    year = str(year) #ensure year is string
    start_date = pd.Timestamp(year+'-01-01 00:00:00') #jan 1st of year
    end_date = pd.Timestamp(year+'-12-31 23:59:59') #dec 31st of year
    filtered_df = df[df['datetime'].between(start_date,end_date)].reset_index(drop=True) #only pull between these dates from df
    unique_dates=filtered_df.copy()
    unique_dates['date'] = unique_dates.datetime.dt.date #create column of just dates without times
    unique_dates = unique_dates.drop_duplicates(subset=['date'], keep='first').astype(str) #find all unique dates and turn to str
    unique_dates['date'] = unique_dates['date'].str.replace('-', '') #take out all - 
    dates = unique_dates.date #return just the dates as an array 
    return filtered_df, dates


In [5]:
def sat_insit_match3(sat_xr, insit_df):
    ''' takes a satellite xarray and a in situ dataframe and matches datapoinst with same lat/lon/datetime 
        sat_xr: satellite xarray subset to only unique dates from in situ dataframe
        insit_df: in situ dataframe subset to specific year 
        return: insitu dataframe with additional colum for satellite matches 
    '''
    time_indexer = xr.DataArray(insit_df['datetime'], dims='point') #pull all the dates and turn into array
    lat_indexer = xr.DataArray(insit_df['lat'], dims='point') #pull all lats and turn into an array
    lon_indexer = xr.DataArray(insit_df['lon'], dims='point') #pull all lons and turn into an array 

    print("Finding nearest satellite points for all in-situ data at once...")
    #use xarray.sel to gather the same lat, lon, and time points as the in situ dataset from the satellite data
    matched_sat_points = sat_xr.sel(time=time_indexer, lat=lat_indexer,lon=lon_indexer,method='nearest')
    
    print("Filtering matches based on spatial and temporal criteria...")
    #calculate the distance in the matched sat point is from the in situ point.
    distances_km = haversine_distance_vectorized( insit_df['lat'].values,  insit_df['lon'].values, matched_sat_points['lat'].values,
                                                 matched_sat_points['lon'].values)

    #check if the match is on the exact same day.
    time_match_mask = insit_df['datetime'].dt.date.values == pd.to_datetime(matched_sat_points['time'].values).date

    #create mask that only retains data where the dates match. 
    valid_indices = np.where(time_match_mask)[0]
    print(f"Found {len(valid_indices)} valid matches out of {len(insit_df)} points. Now extracting 3x3 grids...")
    
    #initiate columns to put satellite information 
    insit_df['chl_occci'] = np.nan
    insit_df['matched_sat_time'] = pd.NaT
    insit_df['matched_spatial_dist_km'] = np.nan

    #Gather surrounding 3x3 pixels and find the average 
    if len(valid_indices) > 0: #as long as there are points that pass the date mask
        sat_lat_coords = sat_xr['lat'].values ##grab the lat and lon points from WHOLE xarray
        sat_lon_coords = sat_xr['lon'].values

        #loop ONLY over the valid matches to extract the 3x3 grid.
        for idx in valid_indices: 
            #get the coordinates of the valid matched satellite point.
            matched_lat = matched_sat_points['lat'].values[idx]
            matched_lon = matched_sat_points['lon'].values[idx]
            matched_time = matched_sat_points['time'].values[idx]

            #using the coordinates from the valid matches, gather the index value that matches to the whole xarray
            lat_idx = sat_xr.indexes['lat'].get_loc(matched_lat)
            lon_idx = sat_xr.indexes['lon'].get_loc(matched_lon)
            time_idx = sat_xr.indexes['time'].get_loc(matched_time)

            #gather slice information the surrounding lat and lon pixels 
            lat_slice = slice(max(0, lat_idx - 1), min(len(sat_lat_coords), lat_idx + 2))
            lon_slice = slice(max(0, lon_idx - 1), min(len(sat_lon_coords), lon_idx + 2))
            
            #extract the 3x3 grid using isel.
            grid_data = sat_xr['chlor_a'].isel(time=time_idx, lat=lat_slice, lon=lon_slice) #gather the chlorophyll data from surrounding pixels
            grid_values = grid_data.values.flatten() #flatten array
            
            #calculate the mean of the grid
            valid_grid_values = grid_values[~np.isnan(grid_values)] #remove any nans from the array
            grid_mean = np.mean(valid_grid_values) if valid_grid_values.size > 0 else np.nan #as long as the array has a value, find the mean, else set as nan

            #store the results in the DataFrame.
            insit_df.loc[idx, 'chl_occci'] = grid_mean
            insit_df.loc[idx, 'matched_sat_time'] = matched_time
            insit_df.loc[idx, 'matched_spatial_dist_km'] = distances_km[idx]

    print('Finished matching process!')
    return insit_df

In [6]:
def insit_match(insitu_df,year):
    '''
    code to gather and concat all xarray files with matching dates to the in situ dataset and subset to north america
    input: 
        insitu_df: dataframe with datetime, lat, lon, and variable
        year: the specific year to subset the dataframe to 
    returns:
        matched_chl:dataframe subset to that specific year with matched OC-CCI chlorophyll
    '''
    #first, use the function insitu_year to subset the in situ dataframe to specific year. this helps with processing
    insit_year, dates_year= insitu_year(insitu_df,year)
    
    existing_files= []
    for date in sorted(dates_year): #for each date that recorded data 
        file_date = r'CHL\ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-'+str(date)+'-fv6.0.nc' #load in the specific nc file name
        if os.path.exists(file_date): #if the file exists, then add to a list of files
            existing_files.append(file_date)
        else:
            print(f"Warning: File not found {date}: {file_date}") #else, skip 
            
#If there are some issues with corrupted files, comment out the above block from "existing_files" to "print", and uncomment the below block
    #existing_files= []
    #for date in sorted(dates_year):
    #    file_date = r'CHL\ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-'+str(date)+'-fv6.0.nc'
    #    if os.path.exists(file_date): #if the file exists, try to open and load it in. 
    #        try:
    #            temp_ds = xr.open_dataset(file_date, chunks={}) #open the file
     #           _ = temp_ds['chlor_a'].isel(time=0, latitude=0, longitude=0).compute() #load in the data instead of lazy load to make sure it exists.
     #           temp_ds.close() #close the file
     #           existing_files.append(file_date) #if it oppened, append to the list of files 
     #       except Exception as e: #if it didn't open, then skip. 
     #           print(f"Skipping corrupted or unreadable file: {file_date}") #. Error: {e}
                #temp_ds.close()
     #   else:
     #        print(f"Warning: File not found for date {date}: {file_date}") #if the file name doesn't exist, skip 
    
    #concat all satellite .nc files into single file       
    coastal_us_combo= xr.open_mfdataset(existing_files, preprocess=slice_before_concat,combine='by_coords') #open all files, concat into one subset to north america
    print('satellite refiguring complete') 
    test_insitu = insit_year.copy().reset_index(drop=True) #subset dataframe to specific year, reset the index
    matched_chl = sat_insit_match3(coastal_us_combo, test_insitu) #run the satellite matching function
    coastal_us_combo.close() #close the file
    return matched_chl  #matched_chl = dataframe of in situ chlorophyll to OC-CCI chlorophyll 


In [10]:
chl_all = pd.read_excel(r"C:\Users\gianna.milton\Documents\Python\Coastal_chl_final\all_chl.xlsx") #load in concatinated chlorophyll dataset.
chl_all['datetime'] = chl_all['datetime'].apply(pd.to_datetime) #ensure datetime is in pandas format 

In [11]:
chl_test=chl_all.loc[chl_all['depth']<=10]  #since OC-CCI is sattellite data and only records 1 depth, reduce to just surface waters 
#next, since this dataset has flourecence along with extracted, only keep 1 datapoint per datetime, lat, and lon. 
#That way (as shown in next cell with variable 'a'), large sample resolutions don't skew the dataset
chl_test=chl_test.drop_duplicates(subset=['datetime', 'lat', 'lon'], keep='first').reset_index(drop=True) 

#run below code to see what years need to be run. 
unique_dates=chl_test.copy() #copy dataframe to ensure no accidental changes happen to the original dataframe
unique_dates['date'] = unique_dates.datetime.dt.date #create column of just date
unique_dates = unique_dates.drop_duplicates(subset=['date'], keep='first').astype(str).reset_index(drop=True) #find all unique dates and turn to str
unique_dates['date'] = unique_dates['date'].str.replace('-', '') #take out all - 
dates_year = unique_dates.date

In [12]:
a=chl_all.groupby(['datetime','lat','lon']).size().reset_index(name='counts') #how many samples are recorded in each datetime, lat, and lon. 
a.nlargest(4, 'counts')

,datetime,lat,lon,counts
8721,2001-04-10 12:34:00,35.20177,-76.24203,315
26707,2003-04-30 00:51:58,34.15600,-119.94900,300
32321,2003-09-11 03:07:41,34.29800,-119.88500,300
34475,2003-11-05 05:08:44,34.34300,-119.86500,300


In [ ]:
chl_2000 =insit_match(chl_test,2000) #run insit_match to the chlorophyll data, subset to 2000. repeat for all years
chl_2000 = chl_2000.dropna(subset=['chl_occci']) #remove any empty OC-CCI matches, since we only want datapoints that correspond.

In [ ]:
chl_2001 =insit_match(chl_test,2001)
chl_2001 = chl_2001.dropna(subset=['chl_occci']) #remove any empty OC-CCI matches, since we only want datapoints that correspond.

In [ ]:
chl_2002 =insit_match(chl_test,2002)
chl_2002 = chl_2002.dropna(subset=['chl_occci'])

In [ ]:
chl_2003 =insit_match(chl_test,2003)
chl_2003 = chl_2003.dropna(subset=['chl_occci'])

In [ ]:
chl_2004 =insit_match(chl_test,2004)
chl_2004 = chl_2004.dropna(subset=['chl_occci'])

In [ ]:
chl_2005 =insit_match(chl_test,2005)
chl_2005 = chl_2005.dropna(subset=['chl_occci'])

In [ ]:
chl_2006 =insit_match(chl_test,2006)
chl_2006 = chl_2006.dropna(subset=['chl_occci'])

In [ ]:
chl_2007 =insit_match(chl_test,2007)
chl_2007 = chl_2007.dropna(subset=['chl_occci'])

In [ ]:
chl_2008 =insit_match(chl_test,2008)
chl_2008 = chl_2008.dropna(subset=['chl_occci'])

In [ ]:
chl_2009 =insit_match(chl_test,2009)
chl_2009 = chl_2009.dropna(subset=['chl_occci'])

In [ ]:
chl_2010 =insit_match(chl_test,2010)
chl_2010 = chl_2010.dropna(subset=['chl_occci'])

In [ ]:
chl_2011 =insit_match(chl_test,2011)
chl_2011 = chl_2011.dropna(subset=['chl_occci'])

In [ ]:
chl_2012 =insit_match(chl_test,2012)
chl_2012 = chl_2012.dropna(subset=['chl_occci'])

In [ ]:
chl_2013 =insit_match(chl_test,2013)
chl_20103 = chl_2013.dropna(subset=['chl_occci'])

In [ ]:
chl_2014 =insit_match(chl_test,2014)
chl_2014 = chl_2014.dropna(subset=['chl_occci'])

In [ ]:
chl_2015 =insit_match(chl_test,2015)
chl_2015 = chl_2015.dropna(subset=['chl_occci'])

In [ ]:
chl_2016 =insit_match(chl_test,2016)
chl_2016 = chl_2016.dropna(subset=['chl_occci'])

In [ ]:
chl_2017 =insit_match(chl_test,2017)
chl_2017 = chl_2017.dropna(subset=['chl_occci'])

In [ ]:
chl_2018 =insit_match(chl_test,2018)
chl_2018 = chl_2018.dropna(subset=['chl_occci'])

In [ ]:
chl_2020 =insit_match(chl_test,2020)
chl_2020 = chl_2020.dropna(subset=['chl_occci'])

In [ ]:
chl_2021 =insit_match(chl_test,2021)
chl_2021 = chl_2021.dropna(subset=['chl_occci'])

In [ ]:
chl_2022 =insit_match(chl_test,2022)
chl_2022 = chl_2022.dropna(subset=['chl_occci'])

In [ ]:
#there are a few sticky years that were having problems matching, so run the same code just Piecewise to help with processing 

In [ ]:
insit_year, dates_year= insitu_year(chl_test,2025) #subset dataframe to the year

existing_files= []
var = 'chlor_a'
for date in sorted(dates_year):
    file_date = r'CHL\ESACCI-OC-L3S-CHLOR_A-MERGED-1D_DAILY_4km_GEO_PML_OCx-'+str(date)+'-fv6.0.nc'
    if os.path.exists(file_date): #if path exists, try to open it
        try:
            with xr.open_dataset(file_date, chunks={}) as temp_ds:
                _ = temp_ds[var].isel(time=0, lat=0, lon=0).compute()
            existing_files.append(file_date)
        except Exception as e:
            print(f"Skipping corrupted or unreadable file: {file_date}")            
    else:
         print(f"Warning: File not found for date {date}: {file_date}")

In [ ]:
coastal_us_combo = xr.open_mfdataset(existing_files, preprocess=slice_before_concat, combine='by_coords', chunks={'time': 1}, engine='netcdf4')

In [ ]:
ch_occci = coastal_us_combo['chlor_a'].load() #subset the xarray to just the needed variable to reduce loading time

In [ ]:
ch_occci.to_netcdf("occci_chl_a_subset_25.nc") #export the nc file to aid in processing speed


In [ ]:
#once the above 4 cells are run, usually you'll need to clear the memory to aid in processing speeds and reload some things

In [ ]:
ch_occci= xr.open_dataset('occci_chl_a_subset_25.nc').load() #load in the xarray satellite data

In [ ]:
insit_year, dates_year= insitu_year(chl_test,2025) #subset to a given year


In [ ]:
#manually run the matchup code
time_indexer = xr.DataArray(insit_year['datetime'], dims='point')
lat_indexer = xr.DataArray(insit_year['lat'], dims='point')
lon_indexer = xr.DataArray(insit_year['lon'], dims='point')

matched_sat_points = ch_occci.sel(time=time_indexer, lat=lat_indexer, lon=lon_indexer, method='nearest')
distances_km = haversine_distance_vectorized(insit_year['lat'].values, insit_year['lon'].values, matched_sat_points['lat'].values, matched_sat_points['lon'].values)

# check if the match is on the exact same day.
time_match_mask = insit_year['datetime'].dt.date.values == pd.to_datetime(matched_sat_points['time'].values).date

insit_year['chl_occci'] = np.nan
insit_year['matched_sat_time'] = pd.NaT
insit_year['matched_spatial_dist_km'] = np.nan

# Get the indices of only the valid matches.
valid_indices = np.where(time_match_mask)[0]
print(f"Found {len(valid_indices)} valid matches out of {len(insit_year)} points. Now extracting 3x3 grids...")

if len(valid_indices) > 0:
    sat_lat_coords = ch_occci['lat'].values
    sat_lon_coords = ch_occci['lon'].values
    #loop ONLY over the valid matches to extract the 3x3 grid.
    for idx in valid_indices:
        #get the coordinates of the valid matched satellite point.
        matched_lat = matched_sat_points['lat'].values[idx]
        matched_lon = matched_sat_points['lon'].values[idx]
        matched_time = matched_sat_points['time'].values[idx]

        #find the integer index using the fast 'get_loc' method.
        lat_idx = ch_occci.indexes['lat'].get_loc(matched_lat)
        lon_idx = ch_occci.indexes['lon'].get_loc(matched_lon)
        time_idx = ch_occci.indexes['time'].get_loc(matched_time)

        #define the 3x3 window boundaries.
        lat_slice = slice(max(0, lat_idx - 1), min(len(sat_lat_coords), lat_idx + 2))
        lon_slice = slice(max(0, lon_idx - 1), min(len(sat_lon_coords), lon_idx + 2))
        
        #extract the 3x3 grid using integer-based selection (isel). Because of .load() above, this no longer touches the hard drive!
        grid_data = ch_occci['chlor_a'].isel(time=time_idx, lat=lat_slice, lon=lon_slice)
        grid_values = grid_data.values.flatten()
        
        #calculate the mean, ignoring NaN values.
        valid_grid_values = grid_values[~np.isnan(grid_values)]
        grid_mean = np.mean(valid_grid_values) if valid_grid_values.size > 0 else np.nan

        #store the results in the DataFrame.
        insit_year.loc[idx, 'chl_occci'] = grid_mean
        insit_year.loc[idx, 'matched_sat_time'] = matched_time
        insit_year.loc[idx, 'matched_spatial_dist_km'] = distances_km[idx]
chl_2025 = insit_year.dropna(subset=['chl_occci'])

In [ ]:
chl_2025=chl_2025.reset_index(drop=True) #reset the index. 

In [ ]:
#since this dataset takes so long to run, everytime a year is matched added it to the existing matchups and save 
current_matchup = pd.read_excel('sat_matchupCHL2.xlsx')  #load in current matchup
dfs=[current_matchup,chl_2025] #append the dataframe to the existing matched dataframe 
matched_all = pd.concat(dfs).reset_index(drop=True)


In [ ]:
matched_all.to_excel("sat_matchupCHL2.xlsx", index = False) #export dataframe 